# full test

对于完整的FPGA mnist测试，要实现

完整两层 MLP：784 -> 128 -> 10

CrossEntropyLoss

静态量化

Python 写整数推理参考实现

导出 FPGA golden

In [12]:
import os
import json
import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [13]:
DATA_DIR = "./data"
OUT_DIR = "./route_b_output"
MODEL_PATH = os.path.join(OUT_DIR, "mlp_route_b.pth")

os.makedirs(OUT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

INT8_QMIN = -128
INT8_QMAX = 127
INT32_QMIN = -2147483648
INT32_QMAX = 2147483647

# 模型

In [14]:
class MLPRouteB(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128, bias=True)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 10, bias=True)

    def forward(self, x):
        x = x.view(-1, 784)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

# 数据
   训练时这里直接用 ToTensor()，不做 Normalize,,这样更贴近最终硬件部署链路

In [15]:
def get_train_test_loaders(batch_size=128):
    transform = transforms.Compose([
        transforms.ToTensor()
    ])

    train_dataset = datasets.MNIST(DATA_DIR, train=True, download=True, transform=transform)
    test_dataset = datasets.MNIST(DATA_DIR, train=False, download=True, transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, test_loader

# 训练 / 测试

In [16]:
def evaluate_model(model, test_loader):
    model.eval()
    total = 0
    correct = 0

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(DEVICE), target.to(DEVICE)
            logits = model(data)
            pred = torch.argmax(logits, dim=1)
            correct += (pred == target).sum().item()
            total += target.size(0)

    return correct / total


def train_model(num_epochs=8, lr=1e-3):
    train_loader, test_loader = get_train_test_loaders()

    model = MLPRouteB().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        for data, target in train_loader:
            data, target = data.to(DEVICE), target.to(DEVICE)

            optimizer.zero_grad()
            logits = model(data)
            loss = criterion(logits, target)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * data.size(0)

        avg_loss = running_loss / len(train_loader.dataset)
        acc = evaluate_model(model, test_loader)
        print(f"[Epoch {epoch+1}/{num_epochs}] loss={avg_loss:.6f}, test_acc={acc:.4f}")

    torch.save(model.state_dict(), MODEL_PATH)
    print(f"Model saved to: {MODEL_PATH}")
    return model

# 量化工具

In [17]:
def clamp(v, low, high):
    return max(low, min(high, v))

def quantize_affine_tensor(tensor_fp, scale, zero_point, qmin=-128, qmax=127, dtype=torch.int8):
    q = torch.round(tensor_fp / scale + zero_point)
    q = torch.clamp(q, qmin, qmax)
    return q.to(dtype)

def calc_symmetric_scale(tensor_fp, num_bits=8):
    qmax = (2 ** (num_bits - 1)) - 1
    max_abs = max(abs(tensor_fp.min().item()), abs(tensor_fp.max().item()))
    if max_abs == 0:
        return 1.0
    return max_abs / qmax

def calc_affine_params_from_range(min_val, max_val, num_bits=8, signed=True):
    if signed:
        qmin = -(2 ** (num_bits - 1))
        qmax =  (2 ** (num_bits - 1)) - 1
    else:
        qmin = 0
        qmax = (2 ** num_bits) - 1

    if max_val == min_val:
        scale = 1.0
        zero_point = 0
        return scale, zero_point

    scale = (max_val - min_val) / float(qmax - qmin)
    zero_point = round(qmin - min_val / scale)
    zero_point = int(clamp(zero_point, qmin, qmax))
    return float(scale), int(zero_point)

def quantize_weight_symmetric_int8(weight_fp):
    scale = calc_symmetric_scale(weight_fp, num_bits=8)
    zp = 0
    q = quantize_affine_tensor(weight_fp, scale, zp, INT8_QMIN, INT8_QMAX, torch.int8)
    return q, scale, zp

def quantize_bias_int32(bias_fp, input_scale, weight_scale):
    bias_scale = input_scale * weight_scale
    q = torch.round(bias_fp / bias_scale)
    q = torch.clamp(q, INT32_QMIN, INT32_QMAX).to(torch.int32)
    return q, bias_scale

def quantize_input_hw_style(img_fp_01):
    """
    固定输入量化：
        x_u8 = round(x * 255)
        x_q  = x_u8 - 128
    对应：
        scale = 1/255
        zero_point = -128
    """
    scale = 1.0 / 255.0
    zp = -128

    x_u8 = torch.round(img_fp_01 * 255.0)
    x_u8 = torch.clamp(x_u8, 0, 255)
    x_q = x_u8.to(torch.int16) - 128
    x_q = torch.clamp(x_q, -128, 127).to(torch.int8)
    return x_q, scale, zp

def choose_output_scale_from_activation(act_fp):
    """
    输出激活采用对称量化到 int8
    """
    scale = calc_symmetric_scale(act_fp, num_bits=8)
    zp = 0
    return scale, zp

def requantize_from_int32(acc_int32, real_multiplier, out_zero_point=0):
    """
    简单软件参考：
        y_q = round(acc * real_multiplier) + out_zero_point
    """
    y = torch.round(acc_int32.to(torch.float32) * real_multiplier) + out_zero_point
    y = torch.clamp(y, INT8_QMIN, INT8_QMAX)
    return y.to(torch.int8)

def quantize_multiplier_to_int(real_multiplier):
    """
    给 FPGA 做定点乘法时可用的近似参数：
      real_multiplier ≈ M / 2^shift
    返回:
      M(int), shift(int)
    这里只提供给配置文件参考；软件 golden 仍直接用 float real_multiplier
    """
    if real_multiplier == 0:
        return 0, 0

    shift = 0
    m = real_multiplier
    while m < 0.5:
        m *= 2.0
        shift += 1
    M = int(round(m * (1 << 31)))  # Q31
    return M, shift + 31

# 量化模型准备

In [18]:
def build_quantized_model(model_fp):
    model_fp.eval()
    model_fp = model_fp.cpu()

    # 固定输入量化参数
    input_scale = 1.0 / 255.0
    input_zero_point = -128

    # fc1 量化
    fc1_w_fp = model_fp.fc1.weight.data
    fc1_b_fp = model_fp.fc1.bias.data
    fc1_w_q, fc1_w_scale, fc1_w_zp = quantize_weight_symmetric_int8(fc1_w_fp)
    fc1_b_q, fc1_b_scale = quantize_bias_int32(fc1_b_fp, input_scale, fc1_w_scale)

    # 用一批训练样本统计 fc1 输出尺度
    calib_loader, _ = get_train_test_loaders(batch_size=256)
    calib_data, _ = next(iter(calib_loader))
    calib_data = calib_data.view(-1, 784)
    with torch.no_grad():
        fc1_act_fp = model_fp.relu(model_fp.fc1(calib_data))
    fc1_out_scale, fc1_out_zp = choose_output_scale_from_activation(fc1_act_fp)

    fc1_real_multiplier = (input_scale * fc1_w_scale) / fc1_out_scale
    fc1_M, fc1_shift = quantize_multiplier_to_int(fc1_real_multiplier)

    # fc2 量化
    fc2_w_fp = model_fp.fc2.weight.data
    fc2_b_fp = model_fp.fc2.bias.data
    fc2_w_q, fc2_w_scale, fc2_w_zp = quantize_weight_symmetric_int8(fc2_w_fp)
    fc2_b_q, fc2_b_scale = quantize_bias_int32(fc2_b_fp, fc1_out_scale, fc2_w_scale)

    # 用一批样本统计 fc2 输出尺度
    with torch.no_grad():
        logits_fp = model_fp(calib_data.view(-1, 1, 28, 28))
    fc2_out_scale, fc2_out_zp = choose_output_scale_from_activation(logits_fp)

    fc2_real_multiplier = (fc1_out_scale * fc2_w_scale) / fc2_out_scale
    fc2_M, fc2_shift = quantize_multiplier_to_int(fc2_real_multiplier)

    qparams = {
        "input": {
            "scale": input_scale,
            "zero_point": input_zero_point,
            "dtype": "int8"
        },
        "fc1": {
            "weight_scale": fc1_w_scale,
            "weight_zero_point": fc1_w_zp,
            "bias_scale": fc1_b_scale,
            "output_scale": fc1_out_scale,
            "output_zero_point": fc1_out_zp,
            "real_multiplier": fc1_real_multiplier,
            "multiplier_q31": fc1_M,
            "shift": fc1_shift
        },
        "fc2": {
            "weight_scale": fc2_w_scale,
            "weight_zero_point": fc2_w_zp,
            "bias_scale": fc2_b_scale,
            "output_scale": fc2_out_scale,
            "output_zero_point": fc2_out_zp,
            "real_multiplier": fc2_real_multiplier,
            "multiplier_q31": fc2_M,
            "shift": fc2_shift
        }
    }

    quantized = {
        "fc1_w_q": fc1_w_q.to(torch.int8),
        "fc1_b_q": fc1_b_q.to(torch.int32),
        "fc2_w_q": fc2_w_q.to(torch.int8),
        "fc2_b_q": fc2_b_q.to(torch.int32),
        "qparams": qparams
    }
    return quantized

# 整数推理

In [19]:
def linear_int8_int32(x_q, w_q, b_q, x_zp, w_zp):
    """
    x_q: [in_features] int8
    w_q: [out_features, in_features] int8
    b_q: [out_features] int32
    """
    x_int = x_q.to(torch.int32) - int(x_zp)
    w_int = w_q.to(torch.int32) - int(w_zp)
    acc = torch.matmul(w_int, x_int) + b_q
    return acc.to(torch.int32)

def relu_int32(x):
    return torch.clamp(x, min=0).to(torch.int32)

def integer_inference_reference(img_fp_01, quantized):
    """
    完整整数参考链路：
      input -> fc1(int32 acc) -> relu -> requant to int8
            -> fc2(int32 acc) -> requant to int8 logits
    """
    qparams = quantized["qparams"]

    # 1) 输入量化
    x_q, _, _ = quantize_input_hw_style(img_fp_01)

    # 2) fc1 int32 累加
    fc1_acc = linear_int8_int32(
        x_q,
        quantized["fc1_w_q"],
        quantized["fc1_b_q"],
        x_zp=qparams["input"]["zero_point"],
        w_zp=qparams["fc1"]["weight_zero_point"]
    )

    # 3) ReLU
    fc1_relu = relu_int32(fc1_acc)

    # 4) fc1 输出 requant -> int8
    fc1_out_q = requantize_from_int32(
        fc1_relu,
        qparams["fc1"]["real_multiplier"],
        qparams["fc1"]["output_zero_point"]
    )

    # 5) fc2 int32 累加
    fc2_acc = linear_int8_int32(
        fc1_out_q,
        quantized["fc2_w_q"],
        quantized["fc2_b_q"],
        x_zp=qparams["fc1"]["output_zero_point"],
        w_zp=qparams["fc2"]["weight_zero_point"]
    )

    # 6) 输出 requant -> int8 logits
    logits_q = requantize_from_int32(
        fc2_acc,
        qparams["fc2"]["real_multiplier"],
        qparams["fc2"]["output_zero_point"]
    )

    pred = int(torch.argmax(logits_q).item())

    return {
        "input_q": x_q,
        "fc1_acc_int32": fc1_acc,
        "fc1_relu_int32": fc1_relu,
        "fc1_out_q": fc1_out_q,
        "fc2_acc_int32": fc2_acc,
        "logits_q": logits_q,
        "pred": pred
    }


# HEX 导出

In [20]:
def write_int8_hex_1d(tensor, path):
    with open(path, "w") as f:
        for v in tensor.flatten():
            f.write(f"{int(v.item()) & 0xFF:02x}\n")

def write_int32_hex_1d(tensor, path):
    with open(path, "w") as f:
        for v in tensor.flatten():
            f.write(f"{int(v.item()) & 0xFFFFFFFF:08x}\n")

def write_uint32_hex_list(values, filename):
    with open(filename, "w") as f:
        for v in values:
            f.write(f"{int(v) & 0xFFFFFFFF:08x}\n")


def export_all_artifacts(model_fp, quantized, num_samples=20):
    # 1) 导出权重 / bias
    # 按 [out][in] 顺序导出
    write_int8_hex_1d(quantized["fc1_w_q"].reshape(-1), os.path.join(OUT_DIR, "fc1_weight_int8.hex"))
    write_int32_hex_1d(quantized["fc1_b_q"], os.path.join(OUT_DIR, "fc1_bias_int32.hex"))

    write_int8_hex_1d(quantized["fc2_w_q"].reshape(-1), os.path.join(OUT_DIR, "fc2_weight_int8.hex"))
    write_int32_hex_1d(quantized["fc2_b_q"], os.path.join(OUT_DIR, "fc2_bias_int32.hex"))

    # 2) 保存量化配置
    config = {
        "network": "784 -> 128 -> 10",
        "weight_layout": {
            "fc1": "row-major [out][in] = [128][784]",
            "fc2": "row-major [out][in] = [10][1288]"
        },
        "input_layout": "[784]",
        "hidden_layout": "[128]",
        "output_layout": "[10]",
        "qparams": quantized["qparams"]
    }

    with open(os.path.join(OUT_DIR, "quant_config.json"), "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2, ensure_ascii=False)

    # 3) 导出样本和 golden
    test_transform = transforms.Compose([
        transforms.ToTensor()
    ])
    test_dataset = datasets.MNIST(DATA_DIR, train=False, download=True, transform=test_transform)

    labels = []
    preds = []

    for i in range(num_samples):
        img_fp, label = test_dataset[i]
        img_flat = img_fp.view(-1).cpu()

        result = integer_inference_reference(img_flat, quantized)

        labels.append(int(label))
        preds.append(int(result["pred"]))

        write_int8_hex_1d(result["input_q"], os.path.join(OUT_DIR, f"input_{i}.hex"))

        write_int32_hex_1d(result["fc1_acc_int32"], os.path.join(OUT_DIR, f"fc1_acc_{i}.hex"))
        write_int32_hex_1d(result["fc1_relu_int32"], os.path.join(OUT_DIR, f"fc1_relu_{i}.hex"))
        write_int8_hex_1d(result["fc1_out_q"], os.path.join(OUT_DIR, f"fc1_out_{i}.hex"))

        write_int32_hex_1d(result["fc2_acc_int32"], os.path.join(OUT_DIR, f"fc2_acc_{i}.hex"))
        write_int8_hex_1d(result["logits_q"], os.path.join(OUT_DIR, f"logits_{i}.hex"))

        with open(os.path.join(OUT_DIR, f"pred_{i}.txt"), "w") as f:
            f.write(str(result["pred"]))

        print(f"sample={i}, label={label}, pred={result['pred']}")

    with open(os.path.join(OUT_DIR, "labels.txt"), "w") as f:
        for lb in labels:
            f.write(f"{lb}\n")

    with open(os.path.join(OUT_DIR, "preds.txt"), "w") as f:
        for pd in preds:
            f.write(f"{pd}\n")

    print("All artifacts exported.")

# 验证量化后整数推理精度

In [21]:
def evaluate_integer_pipeline(model_fp, quantized, num_samples=1000):
    test_transform = transforms.Compose([
        transforms.ToTensor()
    ])
    test_dataset = datasets.MNIST(DATA_DIR, train=False, download=True, transform=test_transform)

    correct = 0
    total = min(num_samples, len(test_dataset))

    for i in range(total):
        img_fp, label = test_dataset[i]
        img_flat = img_fp.view(-1).cpu()
        result = integer_inference_reference(img_flat, quantized)
        if result["pred"] == int(label):
            correct += 1

    acc = correct / total
    print(f"Integer pipeline accuracy over {total} samples: {acc:.4f}")
    return acc

# 运行

In [22]:
if __name__ == "__main__":
    # 1) 训练浮点模型
    model_fp = train_model(num_epochs=8, lr=1e-3)

    # 2) 加载模型（防止单独运行导出时也能用）
    model_fp = MLPRouteB().to(DEVICE)
    model_fp.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model_fp.eval()

    # 3) 构建量化模型
    quantized = build_quantized_model(model_fp)

    qparams = quantized["qparams"]

    quant_param_words = [
        qparams["fc1"]["multiplier_q31"],
        qparams["fc1"]["shift"],
        qparams["fc2"]["multiplier_q31"],
        qparams["fc2"]["shift"],
    ]

    write_uint32_hex_list(
        quant_param_words,
        os.path.join(OUT_DIR, "quant_params.hex")
    )


    # 4) 评估整数链路
    evaluate_integer_pipeline(model_fp, quantized, num_samples=1000)

    # 5) 导出 FPGA 所需文件
    export_all_artifacts(model_fp, quantized, num_samples=20)

[Epoch 1/8] loss=0.668241, test_acc=0.9110
[Epoch 2/8] loss=0.313055, test_acc=0.9217
[Epoch 3/8] loss=0.277090, test_acc=0.9258
[Epoch 4/8] loss=0.257049, test_acc=0.9296
[Epoch 5/8] loss=0.242521, test_acc=0.9314
[Epoch 6/8] loss=0.231383, test_acc=0.9318
[Epoch 7/8] loss=0.222272, test_acc=0.9367
[Epoch 8/8] loss=0.214826, test_acc=0.9392
Model saved to: ./route_b_output/mlp_route_b.pth
Integer pipeline accuracy over 1000 samples: 0.9350
sample=0, label=7, pred=7
sample=1, label=2, pred=2
sample=2, label=1, pred=1
sample=3, label=0, pred=0
sample=4, label=4, pred=4
sample=5, label=1, pred=1
sample=6, label=4, pred=4
sample=7, label=9, pred=9
sample=8, label=5, pred=6
sample=9, label=9, pred=9
sample=10, label=0, pred=0
sample=11, label=6, pred=6
sample=12, label=9, pred=9
sample=13, label=0, pred=0
sample=14, label=1, pred=1
sample=15, label=5, pred=5
sample=16, label=9, pred=9
sample=17, label=7, pred=7
sample=18, label=3, pred=3
sample=19, label=4, pred=4
All artifacts exported.
